<a href="https://colab.research.google.com/github/Asuskf/from-nlp-to-agents/blob/embeddings/embeddings/Active_Structuring/Active_Structuring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab: Active Structuring — From Unstructured Data to Actionable Semantic Pipelines

> **Learn how AI systems transform unstructured information into structured, production-ready data using semantic vectors, dot products, and mathematical inference.**

---

## 🚀 What You'll Build

In this lab, you'll build a semantic processing pipeline that simulates one of the final stages of a Large Language Model.

Instead of relying on keyword matching or regular expressions, you'll use vector representations and the **Dot Product** to determine whether a document contains specific semantic concepts. Finally, you'll convert those mathematical activations into a structured JSON schema that downstream applications or databases can consume.

This workflow mirrors how modern AI systems extract actionable information from unstructured text such as medical reports, support tickets, source code repositories, or business documents.

---

# What You'll Learn

1. **Semantic Projection (Dot Product):** Implement the dot product to mathematically measure how strongly a document activates a target semantic concept within an embedding space.

2. **Active Structuring:** Build an inference pipeline that converts unstructured semantic vectors into a structured JSON schema using geometric thresholds instead of keyword matching.

---

# Lab Overview

We'll progress through three levels:

1. **The Chaos vs. The Schema**
2. **Semantic Projection**
3. **Active Structuring Pipeline**

---

# Level 0 — The Chaos vs. The Schema

Imagine receiving completely unstructured information such as:

- A clinical report
- A Git commit log
- A customer support ticket

Our objective is to automatically determine whether the document belongs to predefined semantic categories.

Instead of searching for words like `"diabetes"` or `"security"`, we'll compare the document's embedding against semantic anchors.


In [1]:
import numpy as np

# Semantic Space:
# [Health_Metrics, System_Logic, Urgency]

schema_anchors = {
    "diabetes_profile": np.array([0.95, 0.05, 0.60]),
    "requires_code_audit": np.array([0.05, 0.95, 0.80])
}

# Simulated embedded documents

doc_a_medical = np.array([0.90, 0.10, 0.50])

doc_b_repo = np.array([0.10, 0.90, 0.90])

# Level 1 — Semantic Projection

To determine how strongly a document represents a target concept, we'll compute the **Dot Product**.

Unlike **Cosine Similarity**, which measures the angle between two vectors, the **Dot Product** measures how much one vector projects onto another while considering both direction and magnitude.

The activation score is computed as:

$$
\text{Activation}=\sum_{i=1}^{n} d_i \times a_i
$$

where:

- $d_i$ represents the document embedding.
- $a_i$ represents the semantic anchor.

A larger activation score indicates stronger evidence that the document expresses the target concept.

In [2]:
def calculate_activation(doc_vector, anchor_vector):

    activation_score = np.dot(doc_vector, anchor_vector)

    return activation_score


score_doc_a = calculate_activation(
    doc_a_medical,
    schema_anchors["diabetes_profile"]
)

score_doc_b = calculate_activation(
    doc_b_repo,
    schema_anchors["diabetes_profile"]
)

print(f"Medical document activation: {score_doc_a:.4f}")
print(f"Repository activation: {score_doc_b:.4f}")

Medical document activation: 1.1600
Repository activation: 0.6800


Notice that the medical document produces a significantly higher activation because its semantic representation aligns more closely with the **diabetes profile** anchor.


# Level 2 — Active Structuring Pipeline

Now we'll transform semantic activations into structured information.

Instead of returning raw numerical scores, we'll automatically generate a JSON object that can be stored in a database or consumed by downstream services.


In [4]:
def active_structuring_pipeline(doc_vector, doc_name, threshold=0.75):

    structured_output = {
        "document_id": doc_name,
        "is_diabetes_profile": False,
        "requires_code_audit": False,
        "confidence_scores": {}
    }

    for concept_name, anchor_vector in schema_anchors.items():

        score = calculate_activation(doc_vector, anchor_vector)

        structured_output["confidence_scores"][concept_name] = round(score, 4)

        if score >= threshold:

            if concept_name == "diabetes_profile":
                structured_output["is_diabetes_profile"] = True

            elif concept_name == "requires_code_audit":
                structured_output["requires_code_audit"] = True

    return structured_output


structured_doc_a = active_structuring_pipeline(
    doc_a_medical,
    "Medical_Record_001"
)

structured_doc_b = active_structuring_pipeline(
    doc_b_repo,
    "Commit_Log_XYZ"
)

import json

print(json.dumps(structured_doc_a, indent=2))
print(json.dumps(structured_doc_b, indent=2))

{
  "document_id": "Medical_Record_001",
  "is_diabetes_profile": true,
  "requires_code_audit": false,
  "confidence_scores": {
    "diabetes_profile": 1.16,
    "requires_code_audit": 0.54
  }
}
{
  "document_id": "Commit_Log_XYZ",
  "is_diabetes_profile": false,
  "requires_code_audit": true,
  "confidence_scores": {
    "diabetes_profile": 0.68,
    "requires_code_audit": 1.58
  }
}
